# 实验一：静态页面与 API 数据采集

- 姓名：李翔宇
- 学号：112404260129
- 班级：数据1241
- GitHub 用户名：xiangyuli88x
- 仓库：https://github.com/xiangyuli88x/bigdata-collection-labs-112404260129
- 平台：https://115.159.159.82


In [1]:
import json
from pathlib import Path
from urllib.parse import urljoin
from collections import Counter

import requests
from bs4 import BeautifulSoup

BASE_URL = "https://115.159.159.82"
session = requests.Session()
print("采集平台：", BASE_URL)

采集平台： https://115.159.159.82


## 任务1：Device 动态加载与 API

`/devices` 的初始 HTML 中只有空表格，浏览器随后请求 `/api/devices` 获取 JSON，所以 Device 属于动态加载。下面复现第一页请求并读取分页结构。

In [2]:
api_url = urljoin(BASE_URL, "/api/devices")
r = session.get(api_url, params={"page": 1, "page_size": 50}, timeout=20)
r.raise_for_status()
data = r.json()

print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))
print("JSON字段:", list(data.keys()))
print("page:", data["page"])
print("page_size:", data["page_size"])
print("total:", data["total"])
print("本页记录数:", len(data["items"]))
print("首条设备:", data["items"][0])

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json
JSON字段: ['page', 'page_size', 'total', 'items']
page: 1
page_size: 50
total: 5000
本页记录数: 50
首条设备: {'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}


## 任务2：自动计算页数并采集全部设备

In [3]:
total = data["total"]
page_size = data["page_size"]
total_pages = (total + page_size - 1) // page_size

devices = []
for page in range(1, total_pages + 1):
    response = session.get(
        api_url,
        params={"page": page, "page_size": page_size},
        timeout=20,
    )
    response.raise_for_status()
    page_data = response.json()
    devices.extend(page_data["items"])

print("每页条数:", page_size)
print("设备总数:", total)
print("计算总页数:", total_pages)
print("实际采集设备数:", len(devices))

每页条数: 50
设备总数: 5000
计算总页数: 100
实际采集设备数: 5000


## 任务3：Company 静态 HTML 解析与3页分页

企业名称直接存在于 `/companies` 的网页源代码中，因此 Company 属于静态加载。企业记录使用 `article.company-card`，下一页使用 `nav.pagination a`。

In [4]:
def parse_company(card):
    details = {}
    for item in card.select("dl > div"):
        key = item.select_one("dt").get_text(strip=True)
        value = item.select_one("dd").get_text(strip=True)
        details[key] = value

    return {
        "company_id": card["data-company-id"],
        "name": card.select_one("h2 a").get_text(strip=True),
        "industry": details["所属行业"],
        "zone": details["所在区域"],
        "device_count": int(details["设备数量"].replace("台", "").strip()),
        "status": card.select_one(".badge").get_text(strip=True),
    }


companies = []
page_counts = []
current_url = urljoin(BASE_URL, "/companies")

for page_number in range(1, 4):
    response = session.get(current_url, timeout=20)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    cards = soup.select("article.company-card")
    page_companies = [parse_company(card) for card in cards]
    companies.extend(page_companies)
    page_counts.append(len(page_companies))

    print(f"第{page_number}页：{len(page_companies)}条，URL={current_url}")

    next_link = next((a for a in soup.select("nav.pagination a")
                      if "下一页" in a.get_text(strip=True)), None)
    if page_number < 3:
        current_url = urljoin(current_url, next_link["href"])

print("Company总数:", len(companies))
print("第一页首条企业:", companies[0])

第1页：20条，URL=https://115.159.159.82/companies
第2页：20条，URL=https://115.159.159.82/companies?page=2
第3页：20条，URL=https://115.159.159.82/companies?page=3
Company总数: 60
第一页首条企业: {'company_id': 'C000001', 'name': '临港精工001有限公司', 'industry': '汽车零部件', 'zone': '智造一区', 'device_count': 6, 'status': '运营中'}


## 任务4：数量、唯一性及 Company–Device 一致性验证

In [5]:
unique_company_ids = len({x["company_id"] for x in companies})
unique_device_ids = len({x["device_id"] for x in devices})

device_counts = Counter(x["company_id"] for x in devices)
mismatches = []
for company in companies:
    cid = company["company_id"]
    expected = company["device_count"]
    actual = device_counts[cid]
    if expected != actual:
        mismatches.append({
            "company_id": cid,
            "expected": expected,
            "actual": actual,
        })

print("Company 总数：", len(companies))
print("唯一 company_id 数：", unique_company_ids)
print("Device 总数：", len(devices))
print("唯一 device_id 数：", unique_device_ids)
print("Company–Device 不一致企业数：", len(mismatches))

Company 总数： 60
唯一 company_id 数： 60
Device 总数： 5000
唯一 device_id 数： 5000
Company–Device 不一致企业数： 0


## 任务5：保存JSON结果

In [6]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("已生成：", data_dir / "devices.json")
print("已生成：", data_dir / "companies.json")

已生成： data/devices.json
已生成： data/companies.json
